# Quickstart: Benchmark Analysis

This notebook shows how to load WebSimBench data and produce basic visualisations.
It works with files of any size — the loader streams 4 GB+ files automatically.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src import (
    discover_files,
    load_runs_df,
    load_frames_df,
    compare_methods,
    scaling_summary,
    timing_breakdown,
    apply_style,
    get_method_color,
    save_figure,
    METHOD_ORDER,
)

apply_style()

## 1. Discover available data files

In [ ]:
files = discover_files("../raw-data")
files[["simulation", "category", "size_human"]]

## 2. Load a single file (run-level summaries)

Pick any file from the table above. `load_runs_df()` streams the JSON so it works on 4 GB files.

In [ ]:
# Use a small file for this demo — swap the path for any file
df = load_runs_df("../raw-data/high-agents/fire/fire_benchmark_1772726105721.json")
print(f"Loaded {len(df)} runs")
df[["method", "renderMode", "agentCount", "avgExecutionMs", "avgComputeTime"]].head(10)

## 3. Compare methods — pivot table

In [ ]:
compare_methods(df, "avgExecutionMs", render_mode="cpu")

## 4. Scaling line chart

In [ ]:
import matplotlib.pyplot as plt

cpu_df = df[df["renderMode"] == "cpu"]

fig, ax = plt.subplots()
for method in METHOD_ORDER:
    subset = cpu_df[cpu_df["method"] == method]
    if subset.empty:
        continue
    ax.plot(
        subset["agentCount"],
        subset["avgExecutionMs"],
        label=method,
        color=get_method_color(method),
        marker="o",
    )

ax.set_xlabel("Agent Count")
ax.set_ylabel("Avg Frame Time (ms)")
ax.set_title("High-Agent Fire: Scaling by Method")
ax.legend()
plt.show()

## 5. Timing breakdown (stacked bar)

In [ ]:
breakdown = timing_breakdown(df, agent_count=100000, render_mode="cpu")

fig, ax = plt.subplots()
breakdown.plot.barh(stacked=True, ax=ax)
ax.set_xlabel("Time (ms)")
ax.set_title("Timing Breakdown at 100k Agents")
plt.show()

## 6. Frame-level drill-down

Load individual frame timings for a specific run to see variance.

In [ ]:
frames = load_frames_df(
    "../raw-data/high-agents/fire/fire_benchmark_1772726105721.json",
    methods=["JavaScript", "WebGPU"],
    agent_counts=[100000],
)
print(f"Loaded {len(frames)} frame records")

fig, ax = plt.subplots()
for method in ["JavaScript", "WebGPU"]:
    subset = frames[frames["method"] == method]
    ax.plot(
        subset["frameNumber"],
        subset["computeTime"],
        label=method,
        color=get_method_color(method),
        alpha=0.7,
    )

ax.set_xlabel("Frame Number")
ax.set_ylabel("Compute Time (ms)")
ax.set_title("Frame-level Compute — JS vs WebGPU @ 100k Agents")
ax.legend()
plt.show()